In [0]:
%sql
create or replace table workspace.gold.dim_store as
select
  store_id,
  manager_id,
  manager_name
from
  workspace.silver.store;
--select * from workspace.gold.dim_store;

In [0]:
%sql
create or replace table workspace.gold.dim_order as
select
  order_id,
  technician_id,
  technician_name,
  service_type,
  order_status
from
  workspace.silver.order;
--select * from workspace.gold.order;

In [0]:
%sql
create or replace table workspace.gold.dim_customer_survey as
select
  order_id,
  responded_flag
from
  workspace.silver.customer_survey;

In [0]:
%sql
create or replace table workspace.gold.fact_budget as
select
  ns_store_id,
  month,
  budget_amount
from
  workspace.silver.ns_budget;

In [0]:
%sql
create or replace table workspace.gold.fact_invoice_details as
select
  o.order_id,
  invoice_id,
  store_id,
  estimate_id,
  date(vehicle_in_datetime) as vehicle_in_date,
  date(vehicle_out_datetime) as vehicle_out_date,
  date(planned_work_start_datetime) as planned_work_start_date,
  date(actual_work_start_datetime) as actual_work_start_date,
  date(planned_completion_datetime) as planned_completion_date,
  date(actual_completion_datetime) as actual_completion_date,
  date(promised_delivery_datetime) as promised_delivery_date,
  date(actual_delivery_datetime) as actual_delivery_date,
  delivered_on_time_rating,
  work_quality_rating,
  cleanliness_rating,
  communication_rating,
  overall_satisfaction_rating,
  estimator_id,
  estimator_name,
  version_no as estimate_version_no,
  estimate_amount,
  invoice_amount
from
  workspace.silver.invoice i
    right join workspace.silver.order o
      on i.order_id = o.order_id
    left join workspace.silver.customer_survey c
      on o.order_id = c.order_id
    left join workspace.silver.estimate e
      on o.order_id = e.order_id;
--select * from workspace.gold.fact_invoice_details;

In [0]:
%sql
create or replace table workspace.gold.data_cube as
select
  fi.order_id,
  fi.invoice_id,
  fi.store_id,
  fi.estimate_id,
  fi.vehicle_in_date,
  fi.vehicle_out_date,
  fi.planned_work_start_date,
  fi.actual_work_start_date,
  fi.planned_completion_date,
  fi.actual_completion_date,
  fi.promised_delivery_date,
  fi.actual_delivery_date,
  fi.delivered_on_time_rating,
  fi.work_quality_rating,
  fi.cleanliness_rating,
  fi.communication_rating,
  fi.overall_satisfaction_rating,
  fi.estimator_id,
  fi.estimator_name,
  fi.estimate_version_no,
  fi.estimate_amount,
  fi.invoice_amount,
  s.manager_id,
  s.manager_name,
  o.technician_id,
  o.technician_name,
  o.service_type,
  o.order_status,
  c.responded_flag,
  b.month,
  b.budget_amount
from
  workspace.gold.fact_invoice_details fi
    left join workspace.gold.dim_order o
      on fi.order_id = o.order_id
    left join workspace.gold.dim_store s
      on fi.store_id = s.store_id  
    left join workspace.gold.dim_customer_survey c
      on fi.order_id = c.order_id
    left join workspace.gold.fact_budget b
      on fi.store_id = b.ns_store_id and month(month)=month(fi.vehicle_in_date) ;
select * from workspace.gold.data_cube where order_status not in ("COMPLETED") limit 40;